# Hand-object compact train features video demonstration

This notebook overlays the compact hand-object training result on the original world-camera video. It keeps every original YOLO object box and every original left/right hand box, then emphasizes the best object selected for each hand: left hand in thick red, right hand in thick blue.

In [ ]:
from pathlib import Path
import re
import sys

import cv2
import numpy as np
import pandas as pd
from IPython.display import Video, display

PROJECT_ROOT = Path('/home/julius/projects/Identification-of-daily-activities-feature-extraction')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from upper_features_extraction.hand_object_interaction.hand_object_compact_train_features import (
    DEFAULT_IMAGE_HEIGHT,
    DEFAULT_IMAGE_WIDTH,
    EPS,
    MAX_MATCH_DISTANCE_PX,
    NO_VALID_HAND_OBJECT_CLASS,
    OBJECT_DIAG_MIN_PX,
    SIDES,
    read_hand_detections,
    read_object_detections,
)

DATASET_ROOT = Path('/home/julius/datasets/seminar_object_detection')
COMPACT_FEATURE_CSV = DATASET_ROOT / 'input_train' / 'hand_object_compact_train_features.csv'
OBJECT_ROOT = DATASET_ROOT / 'lower_feature_outcome' / 'yolo11s_coco_object_outcome_csv'
HAND_ROOT = DATASET_ROOT / 'lower_feature_outcome' / 'yolo11s_left_right_hand_outcome_csv'
VIDEO_ROOT = DATASET_ROOT / 'iAssistADL_dataset_export_2026-05-18' / 'world_camera_videos'
OUTPUT_ROOT = DATASET_ROOT / 'video_demonstration' / 'hand_object_compact_train_features'
WORLD_CAMERA_SUFFIX = '_world_camera'

# Set this to a specific recording, or leave as None to use the first compact-feature recording with a matching video.
FULL_ID = None

# None exports the complete source video.
MAX_SECONDS = None

# True exports one complete representative video for each action type in the compact feature CSV.
EXPORT_ONE_FULL_VIDEO_PER_ACTION = True
OVERWRITE = True

In [ ]:
compact = pd.read_csv(COMPACT_FEATURE_CSV)
print(f'compact shape: {compact.shape}')
display(compact.head())

def participant_from_full_id(full_id: str) -> str:
    return '_'.join(full_id.split('_')[:4])

def video_path_for(full_id: str) -> Path:
    return VIDEO_ROOT / participant_from_full_id(full_id) / f'{full_id}{WORLD_CAMERA_SUFFIX}.mp4'

def object_path_for(full_id: str) -> Path:
    return OBJECT_ROOT / f'{full_id}{WORLD_CAMERA_SUFFIX}.csv'

def hand_path_for(full_id: str) -> Path:
    return HAND_ROOT / f'{full_id}{WORLD_CAMERA_SUFFIX}.csv'

def choose_full_id(requested_full_id: str | None = None) -> str:
    if requested_full_id:
        return requested_full_id
    for candidate in compact['full_id'].drop_duplicates():
        if video_path_for(candidate).exists() and object_path_for(candidate).exists() and hand_path_for(candidate).exists():
            return candidate
    raise FileNotFoundError('No compact-feature full_id has matching video/object/hand CSV inputs.')

def action_type_from_task_name(task_name: str) -> str:
    action = re.sub(r'\d+$', '', str(task_name)).strip('_')
    action = action.replace('Teekochen', 'Tee_kochen')
    action = action.replace('Chaos_several_Task_combine', 'Chaos_several_Tasks_combine')
    return action

def choose_one_full_id_per_action(features: pd.DataFrame) -> pd.DataFrame:
    candidates = features.copy()
    candidates['action_type'] = candidates['task_name'].map(action_type_from_task_name)
    candidates['has_inputs'] = candidates['full_id'].map(
        lambda value: video_path_for(value).exists() and object_path_for(value).exists() and hand_path_for(value).exists()
    )
    candidates = candidates[candidates['has_inputs']].copy()
    if candidates.empty:
        raise FileNotFoundError('No action has matching video/object/hand CSV inputs.')
    valid_summary = (
        candidates.groupby(['action_type', 'full_id', 'participant_id', 'task_name'], as_index=False)
        .agg(
            frame_count=('video_frame_idx', 'size'),
            valid_hand_object_frame_count=('has_valid_hand_object', 'sum'),
            valid_left_frame_count=('has_valid_left_hand_object', 'sum'),
            valid_right_frame_count=('has_valid_right_hand_object', 'sum'),
        )
        .sort_values(['action_type', 'valid_hand_object_frame_count', 'frame_count'], ascending=[True, False, False])
    )
    return valid_summary.groupby('action_type', as_index=False, sort=True).head(1).reset_index(drop=True)

full_id = choose_full_id(FULL_ID)
participant_id = participant_from_full_id(full_id)
video_path = video_path_for(full_id)
object_path = object_path_for(full_id)
hand_path = hand_path_for(full_id)
output_path = OUTPUT_ROOT / participant_id / f'{full_id}_compact_hand_object_best_overlay.mp4'

print('full_id:', full_id)
print('video:', video_path)
print('object detections:', object_path)
print('hand detections:', hand_path)
print('output:', output_path)

record_features = compact[compact['full_id'].eq(full_id)].copy()
summary_cols = [
    'video_frame_idx',
    'left_object_1_probability',
    'right_object_1_probability',
    'has_valid_left_hand_object',
    'has_valid_right_hand_object',
    'has_valid_hand_object',
]
display(record_features[summary_cols].query('has_valid_hand_object > 0').head(10))

action_export_plan = choose_one_full_id_per_action(compact)
display(action_export_plan)
ACTION_FULL_IDS = action_export_plan['full_id'].tolist()

In [ ]:
def read_raw_detections(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    for column in ('frame_id', 'class_id', 'confidence', 'x1', 'y1', 'x2', 'y2'):
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors='coerce')
    df = df.dropna(subset=['frame_id', 'confidence', 'x1', 'y1', 'x2', 'y2']).copy()
    df['frame_id'] = df['frame_id'].round().astype(int)
    if 'class_name' not in df.columns:
        df['class_name'] = ''
    return df

def compact_class_for_row(row: pd.Series, side: str) -> str:
    prefix = f'{side}_object_1_class_'
    cols = [col for col in row.index if col.startswith(prefix)]
    if not cols:
        return NO_VALID_HAND_OBJECT_CLASS
    active = row[cols].astype(float).idxmax()
    if float(row[active]) <= 0:
        return NO_VALID_HAND_OBJECT_CLASS
    return active[len(prefix):]

def compact_best_by_frame(features: pd.DataFrame) -> dict[tuple[int, str], dict]:
    out = {}
    for _, row in features.iterrows():
        frame_id = int(row['video_frame_idx'])
        for side in SIDES:
            valid = int(row.get(f'has_valid_{side}_hand_object', 0)) > 0
            out[(frame_id, side)] = {
                'valid': valid,
                'class_id': compact_class_for_row(row, side) if valid else NO_VALID_HAND_OBJECT_CLASS,
                'probability': float(row.get(f'{side}_object_1_probability', 0.0) or 0.0),
            }
    return out

def recompute_best_object_rows(object_path: Path, hand_path: Path) -> pd.DataFrame:
    objects = read_object_detections(object_path, image_width=DEFAULT_IMAGE_WIDTH, image_height=DEFAULT_IMAGE_HEIGHT)
    hands = read_hand_detections(hand_path)
    if objects.empty or hands.empty:
        return pd.DataFrame()
    pairs = hands[hands['hand_side'].isin(SIDES)].merge(objects, on='frame_id', how='inner', sort=False)
    if pairs.empty:
        return pd.DataFrame()
    closest_x = np.minimum(
        np.maximum(pairs['hand_cx'].to_numpy(dtype=float), pairs['object_x1'].to_numpy(dtype=float)),
        pairs['object_x2'].to_numpy(dtype=float),
    )
    closest_y = np.minimum(
        np.maximum(pairs['hand_cy'].to_numpy(dtype=float), pairs['object_y1'].to_numpy(dtype=float)),
        pairs['object_y2'].to_numpy(dtype=float),
    )
    pairs['hand_object_dx_px'] = pairs['hand_cx'].to_numpy(dtype=float) - closest_x
    pairs['hand_object_dy_px'] = pairs['hand_cy'].to_numpy(dtype=float) - closest_y
    pairs['hand_object_distance_px'] = np.hypot(
        pairs['hand_object_dx_px'],
        pairs['hand_object_dy_px'],
    )
    pairs = pairs[pairs['hand_object_distance_px'] <= MAX_MATCH_DISTANCE_PX].copy()
    object_diag_safe = np.maximum(np.hypot(pairs['object_width'], pairs['object_height']), OBJECT_DIAG_MIN_PX)
    pairs['distance_norm'] = pairs['hand_object_distance_px'] / object_diag_safe
    pairs['interaction_score'] = pairs['large_object_score_factor'] * np.exp(-pairs['distance_norm'])
    pairs['score_sum'] = pairs.groupby(['frame_id', 'hand_side'])['interaction_score'].transform('sum')
    pairs = pairs[pairs['score_sum'] > EPS].copy()
    if pairs.empty:
        return pd.DataFrame()
    pairs['posterior'] = pairs['interaction_score'] / pairs['score_sum']
    ranked = pairs.sort_values(
        ['frame_id', 'hand_side', 'posterior', 'hand_object_distance_px'],
        ascending=[True, True, False, True],
    ).copy()
    return ranked.groupby(['frame_id', 'hand_side'], as_index=False, sort=False).head(1).copy()

def prepare_recording_context(selected_full_id: str, verbose: bool = True) -> dict:
    global full_id, participant_id, video_path, object_path, hand_path, output_path, record_features
    global raw_objects, raw_hands, best_rows, compact_best
    global raw_objects_by_frame, raw_hands_by_frame, best_by_frame_side

    full_id = selected_full_id
    participant_id = participant_from_full_id(full_id)
    video_path = video_path_for(full_id)
    object_path = object_path_for(full_id)
    hand_path = hand_path_for(full_id)
    action_type = action_type_from_task_name(compact.loc[compact['full_id'].eq(full_id), 'task_name'].iloc[0])
    output_path = OUTPUT_ROOT / action_type / participant_id / f'{full_id}_compact_hand_object_best_overlay_full.mp4'
    record_features = compact[compact['full_id'].eq(full_id)].copy()

    raw_objects = read_raw_detections(object_path)
    raw_hands = read_raw_detections(hand_path)
    best_rows = recompute_best_object_rows(object_path, hand_path)
    compact_best = compact_best_by_frame(record_features)

    raw_objects_by_frame = {int(fid): group.copy() for fid, group in raw_objects.groupby('frame_id', sort=False)}
    raw_hands_by_frame = {int(fid): group.copy() for fid, group in raw_hands.groupby('frame_id', sort=False)}
    best_by_frame_side = {(int(row.frame_id), row.hand_side): row for _, row in best_rows.iterrows()}

    info = {
        'action_type': action_type,
        'full_id': full_id,
        'video_path': str(video_path),
        'object_path': str(object_path),
        'hand_path': str(hand_path),
        'output_path': str(output_path),
        'raw_object_boxes': len(raw_objects),
        'raw_hand_boxes': len(raw_hands),
        'best_hand_object_rows': len(best_rows),
    }
    if verbose:
        for key, value in info.items():
            print(f'{key}: {value}')
        display(best_rows.head())
    return info

prepare_recording_context(full_id)

In [ ]:
OBJECT_COLOR = (80, 255, 120)       # green, original YOLO object boxes
HAND_COLORS = {
    'left': (40, 220, 255),         # yellow-ish, original hand boxes
    'right': (80, 160, 255),
    'unknown': (220, 220, 220),
}
BEST_COLORS = {
    'left': (0, 0, 255),            # red, emphasized best object for left hand
    'right': (255, 0, 0),           # blue, emphasized best object for right hand
}
TEXT_BG = (20, 20, 20)

COCO_NAMES = [
    'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light',
    'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow',
    'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee',
    'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard',
    'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple',
    'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch',
    'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone',
    'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear',
    'hair drier', 'toothbrush',
]

def class_label(row: pd.Series) -> str:
    name = str(row.get('class_name', '') or '').strip()
    if name:
        return name
    class_id = row.get('class_id', np.nan)
    if pd.notna(class_id):
        class_id = int(round(float(class_id)))
        if 0 <= class_id < len(COCO_NAMES):
            return COCO_NAMES[class_id]
        return f'class {class_id}'
    return 'object'

def hand_side_from_row(row: pd.Series) -> str:
    name = str(row.get('class_name', '') or '').lower()
    if 'left' in name:
        return 'left'
    if 'right' in name:
        return 'right'
    class_id = row.get('class_id', np.nan)
    if pd.isna(class_id):
        return 'unknown'
    class_id = int(round(float(class_id)))
    return 'left' if class_id == 0 else 'right' if class_id == 1 else 'unknown'

def as_box(row: pd.Series, prefix: str = '') -> tuple[int, int, int, int] | None:
    cols = [f'{prefix}{c}' for c in ('x1', 'y1', 'x2', 'y2')]
    vals = []
    for col in cols:
        value = row.get(col, np.nan)
        if pd.isna(value):
            return None
        vals.append(int(round(float(value))))
    x1, y1, x2, y2 = vals
    if x2 <= x1 or y2 <= y1:
        return None
    return x1, y1, x2, y2

def put_text_box(frame: np.ndarray, text: str, xy: tuple[int, int], color, scale: float = 0.48, thickness: int = 1) -> None:
    x, y = xy
    h, w = frame.shape[:2]
    x = max(0, min(x, w - 1))
    y = max(18, min(y, h - 4))
    (tw, th), baseline = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, thickness)
    cv2.rectangle(frame, (x - 3, y - th - baseline - 4), (min(w - 1, x + tw + 3), y + baseline + 3), TEXT_BG, -1)
    cv2.putText(frame, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, scale, color, thickness, cv2.LINE_AA)

def draw_detection_box(frame: np.ndarray, row: pd.Series, color, label: str, thickness: int = 2, prefix: str = '') -> None:
    box = as_box(row, prefix=prefix)
    if box is None:
        return
    x1, y1, x2, y2 = box
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
    put_text_box(frame, label, (x1, y1 - 6), color, scale=0.44 if thickness <= 2 else 0.58, thickness=1)

def draw_frame_overlay(frame: np.ndarray, frame_idx: int) -> None:
    object_rows = raw_objects_by_frame.get(frame_idx)
    if object_rows is not None:
        for _, row in object_rows.sort_values('confidence', ascending=True).iterrows():
            label = f"YOLO {class_label(row)} {float(row['confidence']):.2f}"
            draw_detection_box(frame, row, OBJECT_COLOR, label, thickness=2)

    hand_rows = raw_hands_by_frame.get(frame_idx)
    if hand_rows is not None:
        for _, row in hand_rows.sort_values('confidence', ascending=True).iterrows():
            side = hand_side_from_row(row)
            label = f"{side} hand {float(row['confidence']):.2f}"
            draw_detection_box(frame, row, HAND_COLORS.get(side, HAND_COLORS['unknown']), label, thickness=2)

    for side in SIDES:
        best = best_by_frame_side.get((frame_idx, side))
        compact_info = compact_best.get((frame_idx, side), {'valid': False, 'probability': 0.0, 'class_id': NO_VALID_HAND_OBJECT_CLASS})
        if best is None or not compact_info['valid']:
            continue
        row = pd.Series({
            'object_x1': best['object_x1'],
            'object_y1': best['object_y1'],
            'object_x2': best['object_x2'],
            'object_y2': best['object_y2'],
        })
        class_id = int(best['object_class_id'])
        class_name = COCO_NAMES[class_id] if 0 <= class_id < len(COCO_NAMES) else f'class {class_id}'
        label = f"BEST {side} -> {class_name} p={compact_info['probability']:.2f}"
        draw_detection_box(frame, row, BEST_COLORS[side], label, thickness=6, prefix='object_')

    put_text_box(frame, f'{full_id}  frame {frame_idx}', (12, 28), (255, 255, 255), scale=0.62, thickness=1)

In [ ]:
def render_overlay_video(video_path: Path, output_path: Path, max_seconds: float | None = None, overwrite: bool = False) -> dict:
    if output_path.exists() and not overwrite:
        return {'status': 'skipped_exists', 'output_path': str(output_path)}

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f'Cannot open video: {video_path}')

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 30.0)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    max_frames = total_frames if max_seconds is None else min(total_frames, int(round(max_seconds * fps)))
    if max_frames <= 0:
        max_frames = total_frames or int(round((max_seconds or 30) * fps))

    output_path.parent.mkdir(parents=True, exist_ok=True)
    writer = cv2.VideoWriter(str(output_path), cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
    if not writer.isOpened():
        cap.release()
        raise RuntimeError(f'Cannot open video writer: {output_path}')

    written = 0
    while written < max_frames:
        ok, frame = cap.read()
        if not ok:
            break
        draw_frame_overlay(frame, written)
        writer.write(frame)
        written += 1

    cap.release()
    writer.release()
    return {
        'status': 'ok',
        'output_path': str(output_path),
        'fps': fps,
        'size': (width, height),
        'written_frame_count': written,
        'source_frame_count': total_frames,
    }

render_results = []
if EXPORT_ONE_FULL_VIDEO_PER_ACTION:
    for selected_full_id in ACTION_FULL_IDS:
        context = prepare_recording_context(selected_full_id, verbose=False)
        print(f"rendering {context['action_type']}: {selected_full_id}")
        result = render_overlay_video(video_path, output_path, max_seconds=None, overwrite=OVERWRITE)
        render_results.append({**context, **result})
else:
    context = prepare_recording_context(full_id, verbose=False)
    result = render_overlay_video(video_path, output_path, max_seconds=MAX_SECONDS, overwrite=OVERWRITE)
    render_results.append({**context, **result})

render_results_df = pd.DataFrame(render_results)
display(render_results_df)

In [ ]:
preview_rows = render_results_df[render_results_df['status'].isin(['ok', 'skipped_exists'])]
preview_path = preview_rows['output_path'].iloc[0]
display(Video(preview_path, embed=False, width=960))